# 1T Classifier Contract Replay

This notebook converts classifier scores into the standard strategy contract used by the options backtester. It does not run option retrieval or option selection directly. Downstream option experiments should read `trade_windows.parquet` from the written contract directory.

In [ ]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name != 'quant-orchestrator':
    REPO_ROOT = next(parent for parent in REPO_ROOT.parents if parent.name == 'quant-orchestrator')
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from quant_orchestrator.platforms.backtesting_frameworks import (
    ScoredPanelTopKReplayConfig,
    read_strategy_artifacts,
    replay_scored_panel_top_k,
)

pd.set_option('display.max_columns', 160)
pd.set_option('display.width', 220)
print('repo_root', REPO_ROOT)

## Configuration

In [ ]:
SCORED_PANEL_PATH = None  # Example: REPO_ROOT / 'artifacts/path/to/scored_panel.parquet'
OUTPUT_DIR = REPO_ROOT / 'artifacts' / 'classifier_1t_contract_replay'

SCORE_COL = 'prob_buy'
TOP_K = 5
THRESHOLD = 0.50
INITIAL_BALANCE = 100_000.0
FEE_BPS = 5.0
SLIPPAGE_BPS = 5.0

# The scored panel must contain at least date, symbol, close, and SCORE_COL.
# Scores are shifted one bar by the replay helper so today's position is based on the prior score.
print('output_dir', OUTPUT_DIR)
print('score_col', SCORE_COL, 'top_k', TOP_K, 'threshold', THRESHOLD)

## Load Scored Panel

In [ ]:
if SCORED_PANEL_PATH is not None:
    scored_panel = pd.read_parquet(SCORED_PANEL_PATH)
elif 'scored_panel' not in globals():
    raise RuntimeError('Provide SCORED_PANEL_PATH or define scored_panel before running this notebook.')

scored_panel = scored_panel.copy()
scored_panel['date'] = pd.to_datetime(scored_panel['date'])
scored_panel['symbol'] = scored_panel['symbol'].astype(str).str.upper()
display(scored_panel.head())
print('rows', len(scored_panel), 'symbols', scored_panel['symbol'].nunique())

## Replay Equity Signals

In [ ]:
config = ScoredPanelTopKReplayConfig(
    score_col=SCORE_COL,
    top_k=TOP_K,
    threshold=THRESHOLD,
    initial_balance=INITIAL_BALANCE,
    fee_bps=FEE_BPS,
    slippage_bps=SLIPPAGE_BPS,
    strategy_name='classifier_1t.scored_panel_top_k',
    output_dir=OUTPUT_DIR,
    metadata={'notebook': 'classifier_1t_options_signal_backtest.ipynb'},
)
result = replay_scored_panel_top_k(scored_panel, config=config)
bundle = read_strategy_artifacts(OUTPUT_DIR)

performance = result.summary.get('performance', {})
summary = pd.DataFrame([{
    'strategy': result.summary.get('strategy_name'),
    'score_col': SCORE_COL,
    'top_k': TOP_K,
    'threshold': THRESHOLD,
    'final_equity': performance.get('final_equity'),
    'total_return_pct': performance.get('total_return_pct'),
    'total_return_multiple': performance.get('total_return_multiple'),
    'max_drawdown_pct': performance.get('max_drawdown_pct'),
    'trades': len(result.rule_replay.trade_windows),
    'actions': len(result.rule_replay.action_tape),
    'manifest': str(OUTPUT_DIR / 'strategy_artifacts_manifest.json'),
}])
display(summary)
display(result.rule_replay.equity.tail())

## Contract Outputs

In [ ]:
print('manifest', bundle.manifest_path)
print('feature_panel', bundle.feature_panel_path)
print('scored_panel', bundle.scored_panel_path)
print('action_tape', bundle.action_tape_path)
print('trade_windows', bundle.trade_windows_path)

display(result.rule_replay.action_tape.tail(20))
display(result.rule_replay.trade_windows.tail(20))